In [1]:
#importing and environment setup 
import os
import re 
import math
import numpy as np
import pandas as pd 
import networkx as nx
from collections import Counter, defaultdict
from datasets import load_dataset
import nltk
from nltk.corpus import stopwords


pd.set_option('display.max_colwidth', None)


nltk.download('stopwords', quiet=True)
english_stopwords = set(stopwords.words('english'))
exception_words = {
    'about', 'above', 'after', 'again', 'against', 'are', "aren't", 'because', 'before', 'between', 'both', 'but', 'can', 'doing', 'during', 
    'each', 'few', 'having', 'how', 'if', 'into', 'no', 'nor', 'not', 'or', 'out', 'over', 'than', 'that', 'until', 'up', 'very'
}
scientific_stopwords = english_stopwords - exception_words

In [2]:
#loading and getting clean data
corpus_df = load_dataset('allenai/scifact', 'corpus')['train'].to_pandas()
claims_df = load_dataset('allenai/scifact', 'claims')['train'].to_pandas()

corpus_df['doc_id'] = corpus_df['doc_id'].astype(str).str.strip()
claims_df['evidence_doc_id'] = claims_df['evidence_doc_id'].astype(str).str.strip()

def normalize_text(text):
    if pd.isna(text) or text is None: return ''
    return re.sub(r'\s+', ' ', re.sub(r'[\r\n\t]+', ' ', str(text))).strip()

def make_paragraph(arr):
    if isinstance(arr, (list, np.ndarray)): return ' '.join(str(s).strip() for s in arr)
    return str(arr).strip()

corpus_df['full_text'] = corpus_df['title'].apply(normalize_text) + ". " + corpus_df['abstract'].apply(make_paragraph).apply(normalize_text)
claims_df['clean_claim'] = claims_df['claim'].apply(normalize_text)

corpus_doc_set = set(corpus_df['doc_id'])
eval_claims = claims_df[claims_df['evidence_doc_id'].isin(corpus_doc_set) & (claims_df['evidence_doc_id'] != '')].copy()

print(f"Corpus size: {len(corpus_df)} | Evaluation claims: {len(eval_claims)}")

Corpus size: 5183 | Evaluation claims: 957


In [3]:
#tokensize and grapg of words weight extractor
def tokenize(text):
    tokens = re.findall(r'\b[a-z0-9]+(?:-[a-z0-9]+)*\b', text.lower())
    return [t for t in tokens if t not in scientific_stopwords and len(t)>1]

def extract_gow_term_weights(tokens, window_size=3):
    if not tokens:
        return{}
  
    G = nx.Graph()
    G.add_nodes_from(set(tokens))
    
    num_token = len(tokens)
    for i in range(num_token):
        w_end = min(i + window_size, num_token)
        curr = tokens[i]
        for j in range(i+1, w_end):
            neighbor = tokens[j]
            if curr != neighbor:
                if G.has_edge(curr, neighbor):
                    G[curr][neighbor]['weight'] += 1
                else:
                    G.add_edge(curr, neighbor, weight=1)

    try:
        return nx.pagerank(G, weight='weight', max_iter=50)
    except:
        deg = dict(G.degree(weight='weight'))
        s = sum(deg.values()) or 1.0
        return {k: v/s for k, v in deg.items()}

In [4]:
#computing Corpus level IDF
doc_tokens_list = []
doc_freq = Counter()
N = len(corpus_df)

for text in corpus_df['full_text']:
    t_set = set(tokenize(text))
    doc_tokens_list.append(t_set)
    doc_freq.update(t_set)

idf_scores = {term: math.log((N + 1) / (df + 1)) + 1.0 for term, df in doc_freq.items()}
print('Total vocabulary size', len(idf_scores))


Total vocabulary size 46916


In [5]:
#indexing the corpus with graph of words (TW-IDF)
corpus_gow_index = []

for idx, text in enumerate(corpus_df['full_text']):
    tokens = tokenize(text)
    tw_dict = extract_gow_term_weights(tokens, window_size=3)
    
    tw_idf = {term: tw * idf_scores.get(term, 1.0) for term, tw in tw_dict.items()}
    corpus_gow_index.append(tw_idf)

print("Indexing complete!")

Indexing complete!


In [6]:
#executing retrieval and evaluating top-10 metrics
corpus_doc_ids = corpus_df['doc_id'].tolist()
true_targets = eval_claims['evidence_doc_id'].tolist()
claim_queries = [tokenize(c) for c in eval_claims['clean_claim']]

recall_at_1, recall_at_5, recall_at_10 = 0, 0, 0
mrr = 0.0
total_eval = len(eval_claims)

print("Scoring evaluation claims against Graph-of-Words index...")
for c_idx in range(total_eval):
    q_tokens = claim_queries[c_idx]
    target_id = true_targets[c_idx]
    
    scores = []
    for d_idx, doc_weights in enumerate(corpus_gow_index):
        s = sum(doc_weights.get(q_term, 0.0) for q_term in q_tokens)
        scores.append((s, corpus_doc_ids[d_idx]))
    
    scores.sort(key=lambda x: x[0], reverse=True)
    retrieved_top10 = [doc_id for _, doc_id in scores[:10]]
    
    if target_id in retrieved_top10[:1]: recall_at_1 += 1
    if target_id in retrieved_top10[:5]: recall_at_5 += 1
    if target_id in retrieved_top10[:10]: recall_at_10 += 1
    if target_id in retrieved_top10:
        mrr += 1.0 / (retrieved_top10.index(target_id) + 1)

# Summary scores
r1, r5, r10, mrr_score = recall_at_1 / total_eval, recall_at_5 / total_eval, recall_at_10 / total_eval, mrr / total_eval

gow_metrics_df = pd.DataFrame({
    'Method': ['Graph-of-Words (TW-IDF)'],
    'Total_Claims': [total_eval],
    'Recall@1': [round(r1, 4)],
    'Recall@5': [round(r5, 4)],
    'Recall@10': [round(r10, 4)],
    'MRR@10': [round(mrr_score, 4)]
})

print("GRAPH-OF-WORDS RETRIEVAL PERFORMANCE")
print(gow_metrics_df)

Scoring evaluation claims against Graph-of-Words index...
GRAPH-OF-WORDS RETRIEVAL PERFORMANCE
                    Method  Total_Claims  Recall@1  Recall@5  Recall@10  \
0  Graph-of-Words (TW-IDF)           957    0.4754    0.7367     0.8433   

   MRR@10  
0  0.5843  


In [7]:
# Checking how GoW performs on the exact query that failed Rank 1 in Notebook 03
sample_tokens = claim_queries[0]
sample_target = true_targets[0]

sample_scores = []
for d_idx, doc_weights in enumerate(corpus_gow_index):
    s = sum(doc_weights.get(q_term, 0.0) for q_term in sample_tokens)
    sample_scores.append((s, d_idx))

sample_scores.sort(key=lambda x: x[0], reverse=True)

print(f"Claim: {eval_claims['clean_claim'].iloc[0]}")
print(f"Target Doc ID: {sample_target}")
print("\nTop 3 GoW Retrieved Papers:")
for rank, (score, d_idx) in enumerate(sample_scores[:3], start=1):
    doc = corpus_df.iloc[d_idx]
    hit = " [CORRECT MATCH!]" if doc['doc_id'] == sample_target else ""
    print(f"Rank {rank} (Score: {score:.4f}){hit} -> Doc ID: {doc['doc_id']} | Title: {doc['title']}")

Claim: 1 in 5 million in UK have abnormal PrP positivity.
Target Doc ID: 13734012

Top 3 GoW Retrieved Papers:
Rank 1 (Score: 0.3402) -> Doc ID: 17333231 | Title: The sheddase ADAM10 is a potent modulator of prion disease
Rank 2 (Score: 0.3293) [CORRECT MATCH!] -> Doc ID: 13734012 | Title: Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey
Rank 3 (Score: 0.2611) -> Doc ID: 42240424 | Title: The effects of prion protein proteolysis and disaggregation on the strain properties of hamster scrapie.


In [8]:
# Load dense baseline metrics from Notebook 03
baseline_path = 'results/tables/baseline_dense_retrieval_metrics.csv'
if os.path.exists(baseline_path):
    dense_df = pd.read_csv(baseline_path)
    dense_df['Method'] = 'Dense Bi-Encoder (MiniLM)'
    comparison_table = pd.concat([dense_df[['Method', 'Recall@1', 'Recall@5', 'Recall@10', 'MRR@10']],
                                  gow_metrics_df[['Method', 'Recall@1', 'Recall@5', 'Recall@10', 'MRR@10']]],
                                 ignore_index=True)
else:
    comparison_table = gow_metrics_df



print("BENCHMARK COMPARISON: DENSE BASELINE vs. GRAPH-OF-WORDS")
print(comparison_table)


BENCHMARK COMPARISON: DENSE BASELINE vs. GRAPH-OF-WORDS
                      Method  Recall@1  Recall@5  Recall@10  MRR@10
0  Dense Bi-Encoder (MiniLM)    0.6301    0.8955     0.9415  0.7401
1    Graph-of-Words (TW-IDF)    0.4754    0.7367     0.8433  0.5843
